# Stage 10 (repurposed) — General F&O trend-following 1D status (live refresh, full universe)

**REPURPOSED** (see `trend_universe.py` and README Stage 10 for the
history): the universe is now the full F&O stock+index universe, not the
154-symbol pledgeable-securities list. The strategy is unchanged in spirit —
buy on a fresh 1D Supertrend UP flip, hold, exit on the opposite DOWN flip —
but it's now literal cash-equity ownership for any of these names, not
specifically about monitoring pledged collateral. **No stop-loss**:
`src/stop_loss_test.py` backtested fixed-%, ATR-multiple, and worst-day-based
candidate stop rules against real 1D flip history and found none beat holding
to the natural DOWN-flip exit — every rule's stopped-out flips would have
averaged a POSITIVE outcome if held (consistent with `reversion_test.py`'s
momentum/continuation finding). Exit is the Supertrend DOWN flip only, by
design, not by omission — `buffer_to_flip_pct`/`moved_since_entry_pct` below
remain the informational "how close/how underwater" numbers.

**Corrected design (kept from the original build):** a purely read-only
version can go stale the same way the credit-spread screener's already-
documented gap works - if a symbol flipped after the last fetch, the saved
report keeps showing the old direction/age. This version genuinely refreshes
every symbol before reporting status.

**Second correction - a real regression found while building the Stage 11
pledged-trend-following ranking:** the original version of this cell called
`run_all_instruments(..., force_refresh=True)` with a short 390-day window,
which does a full REPLACE of each symbol's saved raw data, not an update.
Every routine live-screener run was silently throwing away notebook 10's deep
multi-year history and replacing it with just that short window - after
repeated runs, every symbol's saved history had collapsed to ~1 year, and
UP-closed-flip counts per symbol fell to 1-8, nowhere near usable for a
ranking. Fixed by switching to `incremental=True`
(`process_instrument_incremental()` in `trend_1d_pipeline.py`): it EXTENDS
existing saved history with just the recent catch-up window instead of
replacing it outright, so routine checks no longer erode the deep history
notebook 10 built. `from_date` is no longer meaningful in this mode (each
symbol's own last-saved date drives its fetch window) - only `to_date` and
`delay_seconds` still apply.

**Rate limit hit three times in a row** even after the chunk-pacing fix in
`fetch_data.py` and a 1-second between-symbol delay - close-to-the-limit
pacing (~0.35-0.4s spacing, right at the documented 3 req/sec ceiling) isn't
leaving enough real-world margin for network jitter, and repeated failures in
the same session may also mean Kite briefly throttles further after tripping
the limit once. `delay_seconds=1.0` between symbols - real headroom, not the
minimum. If this still fails, it's worth waiting a minute or two before
retrying, in case of a lingering cooldown from the earlier attempts.

## Column reference

**Is the trend intact?**

| Column | Meaning |
|---|---|
| `direction` | The currently OPEN 1D Supertrend flip |
| `trend_age_days` | Calendar days since that flip |
| `spot` · `flip_level` | Latest close, and the Supertrend level that would reverse the trend |
| `buffer_to_flip_pct` | Distance between them. **Small = about to invalidate.** With no stop-loss (see below), this is the single most actionable "how close to the exit" number here |
| `moved_since_entry_pct` | Signed *for* the trend: positive = working, negative = underwater |

**Does the signal actually work on this instrument?**

| Column | Meaning |
|---|---|
| `dir_flips` | Closed flips on the currently open side. Under 10, the edge columns are blank |
| `dir_win_rate` | % of those that moved the way the signal called |
| `dir_median_move` | Median favourable move. Negative = the typical flip goes the *wrong* way |
| `signal_works_here` | `dir_win_rate >= 50` **and** `dir_median_move > 0` |
| `opp_win_rate` · `opp_median_move` | The other direction, for contrast only |

> **Why this matters on this universe.** Measured on the original 154-symbol
> pledged universe (100 instruments with enough history): UP flips won 54.1%
> (+0.9% mean median move, 68% show an edge). DOWN flips won **42.4%** with a
> **−1.3%** mean median move and only **14%** show an edge — the median DOWN
> flip ends with price *higher* than where it flipped. Worth re-checking this
> split on the full F&O universe once enough calendar time has passed, but
> UP-only scoring is the right default in the meantime.

**No stop-loss.** `src/stop_loss_test.py` tested fixed-% (5/8/10/12%),
ATR-multiple (1.5×/2×/2.5×), and worst-day-based candidate stop rules against
real closed UP flips — every single one reduced mean/median return and win
rate versus simply holding to the natural DOWN-flip exit, because stopped-out
flips consistently would have recovered to a profit if held (consistent with
`reversion_test.py`'s momentum/continuation finding, not reversion). Exit is
the Supertrend DOWN flip only.

**Risk / event context**

| Column | Meaning |
|---|---|
| `worst_day` | Largest single close-to-close day against the position, 4yr |
| `entry_day_move` | Move on the day this flip was born |
| `born_on_outlier_day` | `abs(entry_day_move) > 4%` — a flip created by a one-day collapse is a different event, and the duration stats come from ordinary flips |

**Duration context** — descriptive only

| Column | Meaning |
|---|---|
| `median_duration` | This instrument's typical 1D flip length |
| `past_median` · `past_75th_pct` · `past_90th_pct` | Has the open trend outlived that checkpoint? |
| `runway_left_vs_median` · `_p75` · `_p90` | `checkpoint − age` in signed days; negative = past it |

> `reversion_test.py` and `exhaustion_test.py` both found that crossing a duration
> checkpoint does **not** predict a reversal. Not a warning, not an exit signal.

**Universe & score** *(from `trend_universe.py` and notebook 12)*

| Column | Meaning |
|---|---|
| `security_type` | `Stock` or `Index` — display/filter only, not a separate scoring scale |
| `trend_score` | UP-trend-following rank from notebook 12. Blank = thin sample, not a low score |
| `thin_sample` | Fewer than 15 UP-closed flips, so unscored |
| `outlier_reliance_flag` · `gap_risk_flag` | Score depends on a few outlier wins / high gap history |

*All statistics are in-sample and descriptive — what the signal HAS done, not what it will do.*

In [1]:
import sys
sys.path.append("..")

from datetime import datetime, timedelta
from src.auth import get_kite
from src.instruments import build_nse_token_lookup, get_index_token_lookup
from src.trend_universe import load_trend_universe, save_trend_universe
from src.trend_1d_pipeline import run_all_instruments

kite = get_kite()
universe = load_trend_universe(kite=kite)
save_trend_universe(fno_symbol_list=universe[universe.security_type == "Stock"]["symbol"].tolist())
symbol_list = universe["symbol"].tolist()

token_lookup = build_nse_token_lookup(kite)
token_lookup.update(get_index_token_lookup(kite))

# incremental=True - EXTENDS each symbol's existing saved history with just
# the recent catch-up window (per-symbol, derived from its own last-saved
# date) instead of replacing it outright. from_date is unused in this mode,
# kept only because run_all_instruments still accepts it positionally.
#
# delay_seconds bumped well past the 3 req/sec limit's minimum spacing
# (~0.33s) - three failed attempts in a row means it's worth real headroom
# over cutting it close, even though it makes the full run slower.
results = run_all_instruments(
    kite, token_lookup, symbol_list,
    from_date=None,
    to_date=datetime.now(),
    incremental=True,
    delay_seconds=1.0
)
results["status"].value_counts()

Saved 225 symbols (219 stocks, 6 indices) to /Users/hgmac/Documents/3_trading/Quant/config/trend_universe.csv
[1/225] 360ONE — processing...
Fetching 2025-09-22 to 2026-10-22...
Fetching 2026-10-23 to 2026-10-28...
Saved 2019 candles to /Users/hgmac/Documents/3_trading/Quant/data/raw/360ONE_day.csv
Saved full report to /Users/hgmac/Documents/3_trading/Quant/exports/trend_1d/360ONE_full_report.xlsx
[1/225] 360ONE — done
[2/225] ABB — processing...
Fetching 2025-09-22 to 2026-10-22...
Fetching 2026-10-23 to 2026-10-28...
Saved 2925 candles to /Users/hgmac/Documents/3_trading/Quant/data/raw/ABB_day.csv
Saved full report to /Users/hgmac/Documents/3_trading/Quant/exports/trend_1d/ABB_full_report.xlsx
[2/225] ABB — done
[3/225] ABCAPITAL — processing...
Fetching 2025-09-22 to 2026-10-22...
Fetching 2026-10-23 to 2026-10-28...
Saved 2267 candles to /Users/hgmac/Documents/3_trading/Quant/data/raw/ABCAPITAL_day.csv
Saved full report to /Users/hgmac/Documents/3_trading/Quant/exports/trend_1d/ABC

status
SUCCESS    224
FAILED       1
Name: count, dtype: int64

In [2]:
import pandas as pd

# Every table below is wide (30 columns after the trend_score merge)
# and the Shares table alone is 100+ rows — comfortably past pandas' defaults
# (60 rows, and columns wrap rather than showing all 30 side by side), so both
# axes were truncating with "..." in the middle. Nothing was ever missing from
# the underlying data, just not printed. max_columns=None shows every column;
# width=250 stops them wrapping; max_rows=250 covers the largest table here.
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 250)
pd.set_option("display.max_rows", 250)

In [3]:
from src.trend_1d_screener import check_trend_status
import pandas as pd

results = [check_trend_status(s, universe_df=universe) for s in symbol_list]
results = [r for r in results if r is not None]
status = pd.DataFrame(results).sort_values("trend_age_days")
print("SORTED BY: trend_age_days ascending — sets the order the status_up/top25 tables inherit")
print(f"{len(status)} of {len(symbol_list)} instruments have a current status")

No report found for NIFTYFPI - refresh it first
SORTED BY: trend_age_days ascending — sets the order the status_up/top25 tables inherit
224 of 225 instruments have a current status


In [4]:
# Merge in score_for_trend_following()'s ranking (Stage 10/11) -
# answers "which of several same-day UP flips should I prioritize" instead
# of leaving it a guess. Run notebook 12 first if these files don't exist
# yet or look stale.
ranking = pd.read_excel("../exports/RANKING_trend_following.xlsx")
watchlist = pd.read_excel("../exports/WATCHLIST_trend_following.xlsx")

score_cols = ["symbol", "trend_score", "outlier_reliance_flag", "gap_risk_flag"]
ranking_slim = ranking[score_cols]

# Watchlist symbols have no score yet (below the 15 UP-closed-flip floor) -
# kept as NaN/blank rather than 0, so "no score" is never confused with
# "scored low". thin_sample flags which case applies.
watchlist_slim = watchlist[["symbol"]].copy()
for col in ["trend_score", "outlier_reliance_flag", "gap_risk_flag"]:
    watchlist_slim[col] = None
watchlist_slim["thin_sample"] = True

score_lookup = pd.concat([
    ranking_slim.assign(thin_sample=False),
    watchlist_slim
], ignore_index=True)

status = status.merge(score_lookup, on="symbol", how="left")
print(f"Scored: {status['trend_score'].notna().sum()} of {len(status)} — rest are thin-sample (< 15 UP-closed flips)")

Scored: 201 of 224 — rest are thin-sample (< 15 UP-closed flips)


## All instruments — current status

In [5]:
print("SORTED BY: trend_age_days ascending — most recently flipped first")
status

SORTED BY: trend_age_days ascending — most recently flipped first


,symbol,direction,trend_age_days,median_duration,past_median,past_75th_pct,past_90th_pct,runway_left_vs_median,runway_left_vs_p75,runway_left_vs_p90,dir_flips,dir_win_rate,dir_median_move,worst_day,opp_win_rate,opp_median_move,signal_works_here,spot,flip_level,buffer_to_flip_pct,moved_since_entry_pct,entry_day_move,born_on_outlier_day,security_type,trend_score,outlier_reliance_flag,gap_risk_flag,thin_sample
0,GVT&D,UP,0,54.0,False,False,False,54.0,91.2,154.1,30,50.0,3.14,9.15,53.3,1.19,True,4075.00,3591.43,11.87,0.00,2.39,False,Stock,38.6,⚠ RELIES ON OUTLIER WINS (avg-median gap 14.08pp),NaN,False
1,BOSCHLTD,UP,0,45.5,False,False,False,45.5,90.0,142.5,32,65.6,4.54,5.89,65.6,1.78,True,47375.00,42953.57,9.33,0.00,3.89,False,Stock,86.8,NaN,NaN,False
2,TVSMOTOR,UP,0,48.5,False,False,False,48.5,79.8,140.4,30,60.0,4.16,7.45,46.7,-1.06,True,4239.00,3801.78,10.31,0.00,3.72,False,Stock,63.8,⚠ RELIES ON OUTLIER WINS (avg-median gap 9.26pp),NaN,False
3,HEROMOTOCO,UP,0,41.0,False,False,False,41.0,72.5,98.3,40,50.0,0.03,8.03,47.5,-0.04,True,5348.50,4817.92,9.92,0.00,2.49,False,Stock,41.2,⚠ RELIES ON OUTLIER WINS (avg-median gap 3.92pp),NaN,False
4,NAVINFLUOR,UP,0,48.0,False,False,False,48.0,73.0,113.1,35,60.0,5.13,8.52,22.9,-3.76,True,8500.00,7575.94,10.87,0.00,2.98,False,Stock,NaN,NaN,NaN,NaN
5,PGEL,UP,0,35.0,False,False,False,35.0,63.8,127.0,37,54.1,0.09,16.94,51.4,0.22,True,536.10,458.29,14.51,0.00,7.10,True,Stock,24.8,⚠ RELIES ON OUTLIER WINS (avg-median gap 24.91pp),⚠ HIGH GAP RISK (>10.0%),False
6,WAAREEENER,DOWN,1,28.0,False,False,False,27.0,83.0,121.8,6,NaN,NaN,NaN,NaN,NaN,None,2246.40,2493.94,11.02,1.26,-1.90,False,Stock,NaN,NaN,NaN,NaN
7,APLAPOLLO,UP,1,53.0,False,False,False,52.0,105.5,139.5,30,60.0,3.93,6.87,40.0,-2.97,True,2169.60,1984.85,8.52,-2.00,6.55,True,Stock,57.3,⚠ RELIES ON OUTLIER WINS (avg-median gap 16.37pp),NaN,False
8,ANANDRATHI,UP,1,39.0,False,False,False,38.0,83.2,126.0,13,61.5,0.51,6.55,7.7,-4.52,True,2271.00,2130.94,6.17,0.53,1.30,False,Stock,None,None,None,True
9,ATHERENERG,UP,1,43.5,False,False,False,42.5,138.5,198.0,3,NaN,NaN,NaN,NaN,NaN,None,1651.80,1415.22,14.32,-1.33,6.29,True,Stock,None,None,None,True


In [6]:
status_up = status[status["direction"] == "UP"].sort_values("trend_age_days", ascending=True).head(150).reset_index(drop=True)

print("SORTED BY: trend_age_days ascending — most recently flipped UP first")
status_up

SORTED BY: trend_age_days ascending — most recently flipped UP first


,symbol,direction,trend_age_days,median_duration,past_median,past_75th_pct,past_90th_pct,runway_left_vs_median,runway_left_vs_p75,runway_left_vs_p90,dir_flips,dir_win_rate,dir_median_move,worst_day,opp_win_rate,opp_median_move,signal_works_here,spot,flip_level,buffer_to_flip_pct,moved_since_entry_pct,entry_day_move,born_on_outlier_day,security_type,trend_score,outlier_reliance_flag,gap_risk_flag,thin_sample
0,GVT&D,UP,0,54.0,False,False,False,54.0,91.2,154.1,30,50.0,3.14,9.15,53.3,1.19,True,4075.00,3591.43,11.87,0.00,2.39,False,Stock,38.6,⚠ RELIES ON OUTLIER WINS (avg-median gap 14.08pp),NaN,False
1,BOSCHLTD,UP,0,45.5,False,False,False,45.5,90.0,142.5,32,65.6,4.54,5.89,65.6,1.78,True,47375.00,42953.57,9.33,0.00,3.89,False,Stock,86.8,NaN,NaN,False
2,TVSMOTOR,UP,0,48.5,False,False,False,48.5,79.8,140.4,30,60.0,4.16,7.45,46.7,-1.06,True,4239.00,3801.78,10.31,0.00,3.72,False,Stock,63.8,⚠ RELIES ON OUTLIER WINS (avg-median gap 9.26pp),NaN,False
3,HEROMOTOCO,UP,0,41.0,False,False,False,41.0,72.5,98.3,40,50.0,0.03,8.03,47.5,-0.04,True,5348.50,4817.92,9.92,0.00,2.49,False,Stock,41.2,⚠ RELIES ON OUTLIER WINS (avg-median gap 3.92pp),NaN,False
4,NAVINFLUOR,UP,0,48.0,False,False,False,48.0,73.0,113.1,35,60.0,5.13,8.52,22.9,-3.76,True,8500.00,7575.94,10.87,0.00,2.98,False,Stock,NaN,NaN,NaN,NaN
5,PGEL,UP,0,35.0,False,False,False,35.0,63.8,127.0,37,54.1,0.09,16.94,51.4,0.22,True,536.10,458.29,14.51,0.00,7.10,True,Stock,24.8,⚠ RELIES ON OUTLIER WINS (avg-median gap 24.91pp),⚠ HIGH GAP RISK (>10.0%),False
6,APLAPOLLO,UP,1,53.0,False,False,False,52.0,105.5,139.5,30,60.0,3.93,6.87,40.0,-2.97,True,2169.60,1984.85,8.52,-2.00,6.55,True,Stock,57.3,⚠ RELIES ON OUTLIER WINS (avg-median gap 16.37pp),NaN,False
7,ANANDRATHI,UP,1,39.0,False,False,False,38.0,83.2,126.0,13,61.5,0.51,6.55,7.7,-4.52,True,2271.00,2130.94,6.17,0.53,1.30,False,Stock,None,None,None,True
8,ATHERENERG,UP,1,43.5,False,False,False,42.5,138.5,198.0,3,NaN,NaN,NaN,NaN,NaN,None,1651.80,1415.22,14.32,-1.33,6.29,True,Stock,None,None,None,True
9,SONACOMS,UP,1,39.0,False,False,False,38.0,56.8,103.5,18,44.4,-0.58,8.89,44.4,-2.15,False,880.00,770.84,12.40,3.30,8.95,True,Stock,27.0,⚠ RELIES ON OUTLIER WINS (avg-median gap 6.24pp),NaN,False


## Top 25 UP-trending, ranked by trend_score (best first)

Sorted by score, not age — a fresh flip isn't automatically the best pick, and this is the view meant to answer that question directly when several instruments flip UP the same day. `trend_age_days` is still shown for context. Thin-sample names (score is blank) are included but sort to the bottom, not silently dropped.

## Good buy candidates today — the actual daily shortlist

Requires `signal_works_here == True` explicitly — UP has to have
HISTORICALLY WORKED on that specific instrument, not merely be the direction
currently showing. That is the one column from this session's work that
separates a real signal from noise (see the column reference at the top:
DOWN flips work on only 14% of this universe, UP on 68% — but even within
UP, `signal_works_here` still filters out the instruments where it does not
hold). Trend age is NOT part of the filter — a trend that flipped three
weeks ago and is still intact is just as buyable as one from yesterday
(`reversion_test.py`/`exhaustion_test.py` found age doesn't predict anything).

`thin_sample` names (fewer than 15 UP-closed flips → no `trend_score`)
are excluded here rather than sorted to the bottom — with no score, there's no
ranking basis, and this is meant to be a clean go-to list, not a place to
present unranked guesses next to ranked ones.

In [7]:
good_buy_cols = ["symbol", "security_type", "trend_age_days", "trend_score", "dir_win_rate",
                  "dir_median_move", "buffer_to_flip_pct", "moved_since_entry_pct",
                  "worst_day", "outlier_reliance_flag", "gap_risk_flag"]

good_buy_candidates = status[
    (status["direction"] == "UP") &
    (status["signal_works_here"] == True) &
    (status["thin_sample"] == False)
].sort_values("trend_score", ascending=False)[good_buy_cols].reset_index(drop=True)

print(f"{len(good_buy_candidates)} instrument(s) — UP, signal historically works here, ranked")
print("SORTED BY: trend_score descending")
good_buy_candidates

30 instrument(s) — UP, signal historically works here, ranked
SORTED BY: trend_score descending


,symbol,security_type,trend_age_days,trend_score,dir_win_rate,dir_median_move,buffer_to_flip_pct,moved_since_entry_pct,worst_day,outlier_reliance_flag,gap_risk_flag
0,AXISBANK,Stock,13,87.1,65.7,3.41,1.10,-5.53,7.57,⚠ RELIES ON OUTLIER WINS (avg-median gap 3.33pp),NaN
1,BOSCHLTD,Stock,0,86.8,65.6,4.54,9.33,0.00,5.89,NaN,NaN
2,DRREDDY,Stock,37,77.0,61.9,1.66,8.59,11.22,5.58,NaN,NaN
3,NIFTY,Index,12,75.5,59.6,1.27,2.33,-1.93,5.74,NaN,NaN
4,JSWSTEEL,Stock,2,75.0,62.5,2.92,8.65,-1.27,8.49,⚠ RELIES ON OUTLIER WINS (avg-median gap 3.14pp),NaN
5,SENSEX,Index,13,74.6,60.0,0.97,2.27,-1.44,5.94,NaN,NaN
6,COFORGE,Stock,13,73.3,71.0,9.32,9.16,1.18,8.08,⚠ RELIES ON OUTLIER WINS (avg-median gap 6.08pp),⚠ HIGH GAP RISK (>10.0%)
7,HCLTECH,Stock,15,70.0,60.0,1.32,5.55,5.08,5.95,⚠ RELIES ON OUTLIER WINS (avg-median gap 4.49pp),NaN
8,KOTAKBANK,Stock,22,66.0,57.1,1.00,4.97,2.35,6.33,⚠ RELIES ON OUTLIER WINS (avg-median gap 3.53pp),NaN
9,PERSISTENT,Stock,19,65.2,62.9,3.47,6.12,0.87,7.56,⚠ RELIES ON OUTLIER WINS (avg-median gap 10.14pp),NaN


### Top 25 UP-trending, ranked by score

In [8]:
top25_up = status_up[status_up["direction"] == "UP"].sort_values(
    "trend_score", ascending=False, na_position="last"
).head(25)
print("SORTED BY: trend_score descending, thin-sample (blank score) last")
top25_up

SORTED BY: trend_score descending, thin-sample (blank score) last


,symbol,direction,trend_age_days,median_duration,past_median,past_75th_pct,past_90th_pct,runway_left_vs_median,runway_left_vs_p75,runway_left_vs_p90,dir_flips,dir_win_rate,dir_median_move,worst_day,opp_win_rate,opp_median_move,signal_works_here,spot,flip_level,buffer_to_flip_pct,moved_since_entry_pct,entry_day_move,born_on_outlier_day,security_type,trend_score,outlier_reliance_flag,gap_risk_flag,thin_sample
27,AXISBANK,UP,13,52.0,False,False,False,39.0,74.5,98.9,35,65.7,3.41,7.57,37.1,-1.70,True,1216.80,1203.37,1.10,-5.53,1.85,False,Stock,87.1,⚠ RELIES ON OUTLIER WINS (avg-median gap 3.33pp),NaN,False
1,BOSCHLTD,UP,0,45.5,False,False,False,45.5,90.0,142.5,32,65.6,4.54,5.89,65.6,1.78,True,47375.00,42953.57,9.33,0.00,3.89,False,Stock,86.8,NaN,NaN,False
43,DRREDDY,UP,37,42.0,False,False,False,5.0,31.5,52.0,42,61.9,1.66,5.58,40.5,-0.74,True,1332.80,1218.37,8.59,11.22,1.12,False,Stock,77.0,NaN,NaN,False
17,NIFTY,UP,12,31.0,False,False,False,19.0,45.8,79.0,47,59.6,1.27,5.74,42.6,-0.62,True,22698.85,22170.40,2.33,-1.93,0.65,False,Index,75.5,NaN,NaN,False
11,JSWSTEEL,UP,2,42.5,False,False,False,40.5,68.8,105.1,40,62.5,2.92,8.49,27.5,-4.02,True,1244.00,1136.44,8.65,-1.27,6.28,True,Stock,75.0,⚠ RELIES ON OUTLIER WINS (avg-median gap 3.14pp),NaN,False
23,SENSEX,UP,13,31.0,False,False,False,18.0,41.5,77.2,50,60.0,0.97,5.94,42.0,-0.50,True,73246.03,71585.21,2.27,-1.44,2.11,False,Index,74.6,NaN,NaN,False
25,COFORGE,UP,13,55.5,False,False,False,42.5,77.0,135.0,31,71.0,9.32,8.08,48.4,0.00,True,1957.20,1777.95,9.16,1.18,0.33,False,Stock,73.3,⚠ RELIES ON OUTLIER WINS (avg-median gap 6.08pp),⚠ HIGH GAP RISK (>10.0%),False
33,HCLTECH,UP,15,48.0,False,False,False,33.0,63.0,104.1,35,60.0,1.32,5.95,42.9,-2.05,True,1393.30,1316.00,5.55,5.08,8.42,True,Stock,70.0,⚠ RELIES ON OUTLIER WINS (avg-median gap 4.49pp),NaN,False
39,KOTAKBANK,UP,22,45.0,False,False,False,23.0,65.2,98.0,35,57.1,1.00,6.33,37.1,-1.47,True,442.05,420.07,4.97,2.35,3.82,False,Stock,66.0,⚠ RELIES ON OUTLIER WINS (avg-median gap 3.53pp),NaN,False
37,PERSISTENT,UP,19,41.0,False,False,False,22.0,47.0,97.1,35,62.9,3.47,7.56,51.4,0.26,True,5820.00,5463.58,6.12,0.87,5.52,True,Stock,65.2,⚠ RELIES ON OUTLIER WINS (avg-median gap 10.14pp),NaN,False


In [9]:
top25_up_by_age = top25_up.sort_values("trend_age_days", ascending=True)
print("SORTED BY: trend_age_days ascending — same 25 shares as top25_up, just re-ordered by age")
top25_up_by_age

SORTED BY: trend_age_days ascending — same 25 shares as top25_up, just re-ordered by age


,symbol,direction,trend_age_days,median_duration,past_median,past_75th_pct,past_90th_pct,runway_left_vs_median,runway_left_vs_p75,runway_left_vs_p90,dir_flips,dir_win_rate,dir_median_move,worst_day,opp_win_rate,opp_median_move,signal_works_here,spot,flip_level,buffer_to_flip_pct,moved_since_entry_pct,entry_day_move,born_on_outlier_day,security_type,trend_score,outlier_reliance_flag,gap_risk_flag,thin_sample
1,BOSCHLTD,UP,0,45.5,False,False,False,45.5,90.0,142.5,32,65.6,4.54,5.89,65.6,1.78,True,47375.00,42953.57,9.33,0.00,3.89,False,Stock,86.8,NaN,NaN,False
2,TVSMOTOR,UP,0,48.5,False,False,False,48.5,79.8,140.4,30,60.0,4.16,7.45,46.7,-1.06,True,4239.00,3801.78,10.31,0.00,3.72,False,Stock,63.8,⚠ RELIES ON OUTLIER WINS (avg-median gap 9.26pp),NaN,False
6,APLAPOLLO,UP,1,53.0,False,False,False,52.0,105.5,139.5,30,60.0,3.93,6.87,40.0,-2.97,True,2169.60,1984.85,8.52,-2.00,6.55,True,Stock,57.3,⚠ RELIES ON OUTLIER WINS (avg-median gap 16.37pp),NaN,False
11,JSWSTEEL,UP,2,42.5,False,False,False,40.5,68.8,105.1,40,62.5,2.92,8.49,27.5,-4.02,True,1244.00,1136.44,8.65,-1.27,6.28,True,Stock,75.0,⚠ RELIES ON OUTLIER WINS (avg-median gap 3.14pp),NaN,False
12,SHREECEM,UP,2,46.5,False,False,False,44.5,72.8,97.5,37,56.8,0.96,5.88,45.9,-0.15,True,23200.00,21506.89,7.30,-1.44,3.54,False,Stock,64.4,NaN,NaN,False
13,UNIONBANK,UP,6,36.0,False,False,False,30.0,50.5,98.5,44,52.3,1.36,9.76,40.9,-1.08,True,182.22,167.28,8.20,-0.34,0.47,False,Stock,47.5,⚠ RELIES ON OUTLIER WINS (avg-median gap 3.44pp),NaN,False
14,FEDERALBNK,UP,9,39.0,False,False,False,30.0,54.8,83.5,43,55.8,0.61,8.52,44.2,-3.66,True,346.35,316.31,8.67,-0.72,3.21,False,Stock,52.7,⚠ RELIES ON OUTLIER WINS (avg-median gap 5.14pp),NaN,False
18,LTM,UP,12,45.5,False,False,False,33.5,57.8,101.5,33,54.5,1.29,7.80,51.5,0.94,True,4099.90,3886.55,5.20,-6.61,3.66,False,Stock,56.6,⚠ RELIES ON OUTLIER WINS (avg-median gap 7.78pp),NaN,False
22,HAL,UP,12,36.0,False,False,False,24.0,48.8,82.5,33,57.6,0.18,5.51,48.5,-0.50,True,4822.80,4642.64,3.74,-3.33,1.61,False,Stock,47.1,⚠ RELIES ON OUTLIER WINS (avg-median gap 11.24pp),NaN,False
21,EICHERMOT,UP,12,32.5,False,False,False,20.5,49.0,85.9,46,52.2,0.17,7.05,37.0,-2.68,True,7278.00,6915.86,4.98,-0.09,0.82,False,Stock,48.3,⚠ RELIES ON OUTLIER WINS (avg-median gap 3.30pp),NaN,False
